# QLoRA Spam Classifier — Learning by Doing

**Goal:** Fine-tune GPT-2 to classify spam emails using LoRA (Low-Rank Adaptation), and understand every single step along the way.

**What you will learn in this notebook:**
1. Why full fine-tuning is expensive (memory + time)
2. What LoRA is and the exact math behind it
3. What QLoRA adds on top of LoRA
4. How to attach LoRA adapters to GPT-2 using `peft`
5. How to train, evaluate, and run inference
6. How this compares to the baseline full fine-tuning approach

---
**Project structure:**
```
fine-tuning-gpt2-classification/
├── dataset/                    ← spam/ham CSV files
├── notebooks/
│   ├── fine-tuning-gpt2.ipynb  ← baseline: selective layer freezing
│   └── qlora-spam-classifier.ipynb  ← this notebook: LoRA adapters
└── src/qlora/
    ├── config.py    ← all hyperparameters (read this first!)
    ├── dataset.py   ← data loading pipeline
    ├── model.py     ← GPT-2 + LoRA adapter setup
    ├── trainer.py   ← training loop + metrics
    └── train.py     ← entry-point that ties everything together
```

## 1. Background: The Fine-Tuning Problem

### What is a pre-trained language model?

GPT-2 is trained on ~40 GB of internet text to do one thing: **predict the next token**. During this training, its 124 million parameters absorb a remarkable amount of general knowledge: grammar, facts, reasoning patterns, writing styles.

This pre-trained knowledge is stored in the weight matrices throughout the model. Think of the weights as a "compressed representation of the internet".

### Why fine-tune?

GPT-2 knows language, but it doesn't know YOUR task. To classify spam, it needs to learn that phrases like "FREE MONEY", "Click here NOW", "Congratulations winner" are spam signals. This domain-specific knowledge isn't in the pre-trained weights.

Fine-tuning = continuing the training process on your specific dataset to teach the model your task.

### The cost of full fine-tuning

GPT-2 base has **124 million parameters**. In full fine-tuning, every single one is updated each step.

| What's stored in memory during training | Size (fp32) |
|----------------------------------------|-------------|
| Model weights                           | ~500 MB     |
| Gradients (same size as weights)        | ~500 MB     |
| Adam optimizer: first moment (m)        | ~500 MB     |
| Adam optimizer: second moment (v)       | ~500 MB     |
| **Total**                               | **~2 GB**   |

For GPT-2 (124M) that's manageable. But scale to LLaMA-2 (7B parameters) and you need **~112 GB** of GPU memory just for optimizer states. Most researchers don't have that.

**The question LoRA answers:** Can we fine-tune a model to high performance while only storing and updating a tiny fraction of its parameters?

## 2. What is LoRA? (Low-Rank Adaptation)

*Paper: Hu et al., 2021 — "LoRA: Low-Rank Adaptation of Large Language Models"*

### The key insight

Aghajanyan et al. (2020) made a fascinating discovery: **when you fine-tune a large model, the meaningful weight changes ΔW tend to have very low intrinsic rank.**

What does "low rank" mean? A matrix W of shape (768 × 2304) could theoretically require 1,769,472 numbers to describe its update. But if the update has rank-16, you only need ≤16 independent directions to capture it. The rest is redundant.

LoRA exploits this:

### The Math

Instead of directly computing the weight update ΔW, LoRA **factorises** it:

```
ΔW ≈ B × A

where:
  W₀ has shape (d_out × d_in)   e.g. (2304 × 768)
  A  has shape (r    × d_in)   e.g. (  16 × 768)  ← "down-projection"
  B  has shape (d_out × r  )   e.g. (2304 ×  16)  ← "up-projection"
  r  is the rank (a small number, e.g. 16)
```

The modified forward pass:
```
y = W₀ x  +  (α/r) · B A x
    ────       ──────────────
   frozen         LoRA adapter (only A and B are trained)
```

### Why does this dramatically reduce trainable parameters?

```
Layer:  c_attn  (the Q/K/V projection in GPT-2)
Shape:  (768 × 2304)

Full fine-tuning: 768 × 2304 = 1,769,472 parameters to train

LoRA (r=16):      768×16 + 16×2304
                = 12,288 + 36,864
                = 49,152 parameters  ← 97% fewer!
```

### Initialisation trick

- **B is initialised to zeros** → at the start of training, B×A = 0, so the adapter contributes nothing. The model behaves exactly like the frozen pre-trained model. Stable starting point.
- **A is randomly initialised** → gives the gradient a non-zero starting signal so training can begin.

### After training: merging (optional)

Since the final effective weight is `W = W₀ + (α/r)·BA`, you can **merge** the adapter back into the base weight:
```python
merged_model = model.merge_and_unload()
```
This gives you a standard model (no PEFT overhead) with the adapter baked in. Same speed at inference, zero overhead.

## 3. What is QLoRA? (Quantized LoRA)

*Paper: Dettmers et al., 2023 — "QLoRA: Efficient Finetuning of Quantized LLMs"*

QLoRA = **LoRA** (train low-rank adapters) + **Quantization** (store the frozen base model in 4-bit precision).

### Why add quantization?

LoRA reduces the number of **trainable parameters** dramatically. But the frozen base model still has to live in GPU memory (to compute the forward pass). For large models (7B, 13B, 70B params), the frozen weights themselves are the bottleneck.

### How QLoRA quantizes (3 innovations):

**1. NF4 (NormalFloat 4-bit)**
- Normal pre-trained weights follow a bell-curve (normal distribution).
- NF4 defines 16 special float values placed optimally for that distribution.
- Each 32-bit weight is stored as a 4-bit index (0–15) into this table.
- Memory: 32 bits → 4 bits = **8× memory reduction per weight**.

**2. Double Quantization**
- Quantizing weights requires storing "quantization constants" (scale factors).
- These constants are themselves stored in 32-bit floats — a second pass quantizes them to 8-bit.
- Saves an additional ~0.5 bits per parameter on average.

**3. Paged Optimizers**
- Adam's optimizer states (m and v tensors) are 2× the model size each.
- For a 7B model with fp32 optimizer states: 7B × 4 bytes × 2 = 56 GB — more than most GPUs!
- Paged optimizers store optimizer states in CPU RAM and page them to GPU only when needed (like virtual memory in an OS).

### QLoRA result:
A 7B-parameter model that needed 28 GB GPU VRAM with full fp16 fine-tuning can be fine-tuned on a **single 24 GB GPU** using QLoRA, with almost no loss in quality.

### On Apple MPS (our setup):
The `bitsandbytes` library that performs NF4 quantization **requires CUDA**. It does not work on Apple Metal/MPS.

✅ What we do: **plain LoRA** — all the parameter efficiency, without the memory reduction from 4-bit.

For GPT-2 (124M params), this doesn't matter — the model fits easily in RAM either way. The technique is identical. For 7B+ models on limited hardware, the 4-bit quantization is what enables fine-tuning at all.

> **To run true QLoRA** on a CUDA GPU (e.g., Google Colab T4):
> ```python
> # pip install bitsandbytes
> from transformers import BitsAndBytesConfig
> bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
>                                  bnb_4bit_compute_dtype=torch.bfloat16)
> model = GPT2ForSequenceClassification.from_pretrained("gpt2", quantization_config=bnb_config)
> ```

## 4. Setup

Install the required libraries if you don't have them already.

- `transformers` — HuggingFace library for pre-trained models
- `peft` — HuggingFace's PEFT library (implements LoRA, Prefix Tuning, etc.)
- `torch` — PyTorch deep learning framework
- `scikit-learn` — for metrics (accuracy, F1, precision, recall)
- `pandas` — for reading CSV files
- `matplotlib` — for plotting training curves

In [ ]:
# Run this cell once to install required packages.
# The -q flag suppresses verbose output.
# If you're on Colab, these may already be installed.
%pip install -q transformers peft torch scikit-learn pandas matplotlib datasets

## 5. Imports

We import from our `src/qlora/` package. Each module has detailed explanations — open them alongside this notebook!

In [ ]:
# Standard library
import os       # File path utilities
import sys      # Python path manipulation
import time     # Timing training runs

# Third-party
import torch    # PyTorch — the deep learning framework under everything
import numpy as np          # Numerical arrays (used in metrics computation)
import pandas as pd          # DataFrames (used to load and display CSV data)
import matplotlib.pyplot as plt  # Plotting training curves

# ── Tell Python where to find our src/ package ──
# This notebook lives in notebooks/  but our code is in  src/qlora/
# We go one level up (to the project root) so `from src.qlora import ...` works.
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # Prepend so our code takes priority

print(f"Project root: {project_root}")
print(f"Python path includes: {sys.path[0]}")

# ── Import our modules ──
# config.py: the three dataclasses that hold all hyperparameters
from src.qlora.config import LoRAConfig, TrainingConfig, DataConfig

# dataset.py: the SpamDataset class and the data loading pipeline
from src.qlora.dataset import SpamDataset, load_data

# model.py: tokenizer factory, model factory, and parameter counting utility
from src.qlora.model import create_tokenizer, create_model, print_trainable_parameters

# trainer.py: metrics function, TrainingArguments builder, Trainer builder
from src.qlora.trainer import compute_metrics, build_training_args, build_trainer

# train.py: the full pipeline in one call — useful for quick experiments
from src.qlora.train import get_device, train

print("\nAll imports successful!")

## 6. Hardware Check

Our code automatically uses the best available device:
- **CUDA** (NVIDIA GPU): fastest, supports full QLoRA with 4-bit quantization
- **MPS** (Apple Silicon): fast, but no 4-bit quantization support
- **CPU**: always works, but slow

In [ ]:
# get_device() prints what it found and returns a string like "mps" or "cuda"
device = get_device()
print(f"\nSelected device: {device}")

# Quick sanity check: create a small tensor on the device
# torch.zeros(3, device=device) creates a tensor [0., 0., 0.] on the target device.
# If this fails, the device isn't set up correctly.
test_tensor = torch.zeros(3, device=device)
print(f"Test tensor on {device}: {test_tensor}  ✓")

## 7. Configuration

All hyperparameters live in three dataclasses. Think of them as dials you can turn.

### Key LoRA parameters to experiment with:

| Parameter | Default | What changing it does |
|-----------|---------|----------------------|
| `r` (rank) | 16 | Lower → fewer params, less capacity. Higher → more params, more capacity. |
| `lora_alpha` | 32 | Scaling factor. Effective scale = alpha/r. Usually keep = 2×r. |
| `lora_dropout` | 0.1 | Higher → more regularisation (helps with small datasets). |
| `target_modules` | [c_attn, c_proj] | Add c_fc for more capacity; remove c_proj for fewer params. |

### Key training parameters:

| Parameter | Default | Notes |
|-----------|---------|-------|
| `learning_rate` | 2e-4 | Higher than full fine-tuning (2e-5) because we can't destroy frozen weights. |
| `num_train_epochs` | 3 | More epochs = more learning, but watch for overfitting. |
| `per_device_train_batch_size` | 8 | Larger = more stable gradients but more memory. |

In [ ]:
# Instantiate configs with default values.
# To override a value: LoRAConfig(r=8) or LoRAConfig(r=8, lora_alpha=16)
lora_cfg     = LoRAConfig()      # r=16, alpha=32, dropout=0.1, target=[c_attn, c_proj]
training_cfg = TrainingConfig()  # lr=2e-4, epochs=3, batch=8
data_cfg     = DataConfig()      # 1500 samples/class, max_length=128, 80/20 split

# Print the config values so this cell is self-documenting
print("LoRA Config:")
print(f"  rank (r)         = {lora_cfg.r}")
print(f"  alpha            = {lora_cfg.lora_alpha}")
print(f"  effective scale  = {lora_cfg.lora_alpha / lora_cfg.r:.2f}  (alpha / r)")
print(f"  dropout          = {lora_cfg.lora_dropout}")
print(f"  target modules   = {lora_cfg.target_modules}")

print("\nTraining Config:")
print(f"  learning rate    = {training_cfg.learning_rate}")
print(f"  epochs           = {training_cfg.num_train_epochs}")
print(f"  batch size       = {training_cfg.per_device_train_batch_size}")
print(f"  warmup steps     = {training_cfg.warmup_steps}")
print(f"  best metric      = {training_cfg.metric_for_best_model}")

print("\nData Config:")
print(f"  dataset          = {data_cfg.dataset_path}")
print(f"  max token length = {data_cfg.max_length}")
print(f"  samples/class    = {data_cfg.max_samples_per_class}")
print(f"  train/val split  = {data_cfg.train_ratio:.0%} / {1-data_cfg.train_ratio:.0%}")

## 8. Data Loading

### The pipeline
```
spam_ham_dataset.csv
        ↓  pd.read_csv()
  DataFrame (101,594 rows)
        ↓  balance classes (undersample majority)
  Balanced DataFrame (3,000 rows: 1,500 spam + 1,500 ham)
        ↓  train_test_split(stratify=labels)
  train_texts (2,400) + val_texts (600)
        ↓  SpamDataset (lazy tokenization)
  train_dataset + val_dataset   ← ready for Trainer
```

### What is tokenization?

Neural networks only understand numbers. We need to convert text to integers.

GPT-2 uses **Byte-Pair Encoding (BPE)**: it learned a vocabulary of 50,257 "subword" tokens from the training corpus. Common words get a single token; rare words are split into pieces.

```
"Free money! Click here." 
→ ["Free", " money", "!", " Click", " here", "."] (BPE tokens)
→ [11146,   1637,   0,    6914,   994,    13]       (integer IDs)
```

The model's **embedding table** then converts each integer ID into a 768-dimensional vector. That's the first transformation inside GPT-2.

In [ ]:
# create_tokenizer() loads the GPT-2 tokenizer and sets pad_token = eos_token.
# See src/qlora/model.py for the detailed explanation of why we need this patch.
tokenizer = create_tokenizer("gpt2")

In [ ]:
# ── Let's see tokenization in action ──
# This demo shows exactly what happens when we process an email.

example_spam = "CONGRATULATIONS! You've been selected to receive a FREE $1000 gift card! Click NOW!"
example_ham  = "Hey, just wanted to follow up on the meeting notes from yesterday. See you Thursday."

for label, text in [("SPAM", example_spam), ("HAM", example_ham)]:
    print(f"\n{'='*60}")
    print(f"Label: {label}")
    print(f"Text: {text}")
    
    # tokenizer.tokenize() returns the list of string token pieces (NOT IDs yet)
    # This shows how BPE splits the text into subword units
    tokens = tokenizer.tokenize(text)
    print(f"\nTokens ({len(tokens)} total): {tokens}")
    
    # tokenizer.encode() returns the integer IDs for each token
    ids = tokenizer.encode(text)
    print(f"\nToken IDs: {ids}")
    
    # tokenizer() with return_tensors="pt" returns the full encoding dict
    # including attention_mask and padding up to max_length
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=data_cfg.max_length,
        padding="max_length",
        return_tensors="pt"
    )
    
    # The attention_mask is 1 for real tokens and 0 for padding.
    # We count the 1s to see how many tokens are real (vs padded).
    real_tokens = encoding["attention_mask"].sum().item()
    print(f"\nAfter padding to {data_cfg.max_length}:")
    print(f"  input_ids shape:      {encoding['input_ids'].shape}")
    print(f"  attention_mask shape: {encoding['attention_mask'].shape}")
    print(f"  Real tokens:          {real_tokens} / {data_cfg.max_length}")
    print(f"  Padding tokens:       {data_cfg.max_length - real_tokens}")

In [ ]:
# load_data() runs the full pipeline: CSV → balance → split → SpamDataset.
# See src/qlora/dataset.py for line-by-line explanation.

# We need to be in the project root for the relative path in data_cfg to work.
os.chdir(project_root)

train_dataset, val_dataset = load_data(data_cfg, tokenizer)

print(f"\nDataset sizes:")
print(f"  Training:   {len(train_dataset)} examples")
print(f"  Validation: {len(val_dataset)} examples")

In [ ]:
# Let's inspect one sample from the dataset to verify everything looks right.
# SpamDataset.__getitem__(0) returns the first example as a dict of tensors.
sample = train_dataset[0]

print("Sample from train_dataset[0]:")
print(f"  Keys: {list(sample.keys())}")
print(f"\n  input_ids:")
print(f"    shape: {sample['input_ids'].shape}")
print(f"    dtype: {sample['input_ids'].dtype}")
print(f"    first 15 values: {sample['input_ids'][:15].tolist()}")
print(f"    last  15 values: {sample['input_ids'][-15:].tolist()}  ← may see 50256 padding")

print(f"\n  attention_mask:")
print(f"    shape: {sample['attention_mask'].shape}")
print(f"    1s (real tokens): {sample['attention_mask'].sum().item()}")
print(f"    0s (padding):     {(sample['attention_mask'] == 0).sum().item()}")

print(f"\n  label:")
print(f"    value: {sample['labels'].item()}  ({'spam' if sample['labels'].item() == 1 else 'ham'})")
print(f"    dtype: {sample['labels'].dtype}")

# Decode the input_ids back to text to verify
# skip_special_tokens=True removes the padding <|endoftext|> tokens from the output
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
print(f"\n  Decoded text (first 150 chars): '{decoded[:150]}...'") 

## 9. The Model: GPT-2 + LoRA Adapters

### GPT-2 architecture (simplified)

```
Input tokens (128 integers)
       ↓
Token Embedding (50,257 vocab × 768 dims)
Position Embedding (128 positions × 768 dims)
       ↓
Transformer Block × 12  ← each block has:
  LayerNorm
  Multi-Head Self-Attention:
    c_attn: (768 → 2304)  ← Q, K, V combined  ← LoRA goes HERE
    Scaled dot-product attention
    c_proj: (768 → 768)   ← output projection  ← LoRA goes HERE
  LayerNorm  
  Feed-Forward Network:
    c_fc:   (768 → 3072)
    GELU activation
    c_proj: (3072 → 768)
       ↓
Final LayerNorm
       ↓
Last non-padding token's hidden state (768 dims)
       ↓
score: Linear(768 → 2)  ← classification head (fully trained)
       ↓
Logits: [score_ham, score_spam]
```

### Where LoRA attaches

LoRA adapters are added **in parallel** to `c_attn` and `c_proj` in each of the 12 transformer blocks.

That's 12 blocks × 2 layers = **24 adapter pairs (A and B matrices)**.

All other layers (embeddings, feed-forward, layer norms) remain completely frozen.

In [ ]:
# create_model() loads GPT-2 and wraps it with LoRA adapters.
# See src/qlora/model.py for the detailed line-by-line explanation.
model = create_model(lora_cfg, model_name="gpt2")

In [ ]:
# Print the parameter summary — this is the key efficiency metric.
# We expect ~0.5% of parameters to be trainable.
print_trainable_parameters(model)

In [ ]:
# Let's look at the model structure to see where the LoRA adapters are.
# model.print_trainable_parameters() is a built-in peft utility (less detailed than ours).

print("Layers with LoRA adapters (requires_grad=True):")
print("-" * 60)

# model.named_parameters() yields (name, tensor) for every parameter in the model.
# We filter for trainable ones to see exactly what's being updated.
lora_params = [
    (name, param.shape, param.numel())
    for name, param in model.named_parameters()
    if param.requires_grad
]

for name, shape, count in lora_params[:20]:  # Show first 20 to keep output manageable
    print(f"  {name}")
    print(f"    shape: {list(shape)}   params: {count:,}")

if len(lora_params) > 20:
    print(f"  ... and {len(lora_params) - 20} more")

print(f"\nTotal trainable parameter tensors: {len(lora_params)}")
print("\nPattern: lora_A (down-projection) and lora_B (up-projection) per target layer")
print("         plus 'modules_to_save.default' for the classification head (score)")

In [ ]:
# ── Let's verify the LoRA math manually ──
# Find the first c_attn LoRA layer and check the matrix shapes.

print("Verifying LoRA decomposition for the first c_attn layer:")
print("-" * 60)

# Access the first transformer block's attention c_attn layer
# model.base_model.model.transformer.h[0].attn.c_attn is the LoraLayer wrapper
try:
    first_block = model.base_model.model.transformer.h[0]
    c_attn = first_block.attn.c_attn
    
    print(f"  Layer type: {type(c_attn).__name__}")
    
    # The LoraLayer stores the original weight (frozen) and the A/B matrices
    d_in = c_attn.weight.shape[1]   # input dimension = 768
    d_out = c_attn.weight.shape[0]  # output dimension = 2304
    print(f"  Original W₀ shape: {list(c_attn.weight.shape)}  (d_out × d_in = {d_out} × {d_in})")
    print(f"  Original W₀ frozen: {not c_attn.weight.requires_grad}")
    print(f"  Original W₀ params: {c_attn.weight.numel():,}")
    
    # Access the LoRA adapter for the default adapter (key is always 'default')
    lora_A = c_attn.lora_A['default']  # nn.Linear: d_in → r
    lora_B = c_attn.lora_B['default']  # nn.Linear: r → d_out
    
    print(f"\n  lora_A shape: {list(lora_A.weight.shape)}  (r × d_in = {lora_A.weight.shape[0]} × {lora_A.weight.shape[1]})")
    print(f"  lora_B shape: {list(lora_B.weight.shape)}  (d_out × r = {lora_B.weight.shape[0]} × {lora_B.weight.shape[1]})")
    print(f"  lora_A trainable: {lora_A.weight.requires_grad}")
    print(f"  lora_B trainable: {lora_B.weight.requires_grad}")
    
    lora_params_here = lora_A.weight.numel() + lora_B.weight.numel()
    print(f"\n  LoRA params for this layer: {lora_A.weight.numel():,} + {lora_B.weight.numel():,} = {lora_params_here:,}")
    print(f"  Original params for this layer: {c_attn.weight.numel():,}")
    print(f"  Reduction: {c_attn.weight.numel() / lora_params_here:.1f}x fewer params!")
    print(f"\n  Scaling factor (alpha/r): {lora_cfg.lora_alpha}/{lora_cfg.r} = {lora_cfg.lora_alpha/lora_cfg.r}")
    
    # Verify B is zero-initialized
    b_is_zero = (lora_B.weight.data == 0).all().item()
    print(f"\n  lora_B is zero-initialized: {b_is_zero}  ← adapter outputs nothing at start")

except Exception as e:
    print(f"  (Could not access internals directly: {e})")
    print("  This is fine — the peft library may wrap layers differently.")

## 10. Training

### What happens in one training step

```
batch = DataLoader samples 8 emails
       ↓
forward pass: model(input_ids, attention_mask, labels)
       ↓
output.loss = CrossEntropyLoss(logits, labels)
  CrossEntropyLoss = -log(softmax(logit_for_correct_class))
  e.g., if correct class is spam (1) and logits = [0.3, 2.1]:
    softmax → [0.14, 0.86]  (86% confidence in spam)
    loss = -log(0.86) = 0.15  (low loss, model was right)
       ↓
backward pass: loss.backward()
  Compute ∂loss/∂lora_A and ∂loss/∂lora_B for all 24 adapter pairs
  The frozen W₀ receives NO gradient
       ↓
AdamW.step(): update lora_A and lora_B values
       ↓
Repeat for next batch
```

### Why CrossEntropyLoss?

For binary classification, we output two scores: `[score_ham, score_spam]`. CrossEntropyLoss:
1. Applies softmax to convert raw scores to probabilities
2. Takes the negative log of the probability assigned to the CORRECT class
3. A confident correct prediction → low loss. A confident wrong prediction → high loss.

The model is incentivised to be **confidently correct** — not just right, but right with high probability.

In [ ]:
# Build the TrainingArguments and Trainer objects.
# See src/qlora/trainer.py for detailed explanations.

training_args = build_training_args(training_cfg)
trainer = build_trainer(model, train_dataset, val_dataset, training_args)

print("\nTrainer is ready. Components:")
print(f"  Model type:     {type(model).__name__}")
print(f"  Train samples:  {len(train_dataset)}")
print(f"  Val samples:    {len(val_dataset)}")
print(f"  Steps/epoch:    {len(train_dataset) // training_cfg.per_device_train_batch_size}")
print(f"  Total steps:    {len(train_dataset) // training_cfg.per_device_train_batch_size * training_cfg.num_train_epochs}")

In [ ]:
# ── RUN TRAINING ──
# This cell will take 3-8 minutes on Apple Silicon (MPS) or ~1-2 min on a GPU.
#
# Watch the output for:
#   - Loss decreasing over steps (model learning)
#   - Validation metrics improving each epoch (model generalising)
#   - Best checkpoint being saved (load_best_model_at_end=True)

print("Starting training...")
print("Watch the training loss decrease. Validation metrics appear after each epoch.\n")

t_start = time.time()

# trainer.train() is the one call that runs everything.
# It returns a TrainOutput namedtuple.
train_output = trainer.train()

t_end = time.time()
elapsed = t_end - t_start

print(f"\nTraining complete!")
print(f"  Total time:      {elapsed:.1f}s  ({elapsed/60:.1f} min)")
print(f"  Training loss:   {train_output.training_loss:.4f}")
print(f"  Total steps:     {train_output.global_step}")

## 11. Evaluation

### Understanding the metrics

```
Confusion Matrix for Spam Detection:

                  Predicted HAM    Predicted SPAM
Actual HAM    →       TN               FP  ← "False Alarm"
Actual SPAM   →       FN  ← "Missed"   TP

Precision = TP / (TP + FP)   "How many flagged emails were actually spam?"
Recall    = TP / (TP + FN)   "How many actual spam emails did we catch?"
F1        = 2 × P × R / (P + R)   "Balance of precision and recall"
```

In spam filtering, there's a real-world trade-off:
- **High precision, low recall**: rarely wrongly flags ham, but misses a lot of spam
- **High recall, low precision**: catches almost all spam, but generates many false alarms
- **High F1**: balanced — catches most spam without too many false alarms

In [ ]:
# Run the full evaluation loop on the validation set.
# trainer.evaluate() calls compute_metrics() and returns a dict.
eval_results = trainer.evaluate()

print("\nValidation Set Results:")
print("-" * 40)

# The keys have an "eval_" prefix added by Trainer.
metrics = {
    "Accuracy":  eval_results["eval_accuracy"],
    "F1 Score":  eval_results["eval_f1"],
    "Precision": eval_results["eval_precision"],
    "Recall":    eval_results["eval_recall"],
}

for metric, value in metrics.items():
    # Create a simple ASCII bar chart for visual intuition
    bar_length = int(value * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"  {metric:<12} {value:.4f}  |{bar}|")

print(f"\n  Evaluation loss: {eval_results['eval_loss']:.4f}")
print(f"  Runtime: {eval_results.get('eval_runtime', 'N/A'):.1f}s")

In [ ]:
# Plot training curves from the Trainer's logged history.
# trainer.state.log_history is a list of dicts logged at each evaluation/logging step.

log_history = trainer.state.log_history

# Separate training loss entries (logged every logging_steps) from eval entries (logged every epoch)
train_entries = [entry for entry in log_history if "loss" in entry and "eval_loss" not in entry]
eval_entries  = [entry for entry in log_history if "eval_f1" in entry]

if train_entries and eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Training Loss over steps
    steps  = [e["step"]  for e in train_entries]
    losses = [e["loss"]  for e in train_entries]
    axes[0].plot(steps, losses, color="steelblue", linewidth=2, marker="o", markersize=3)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss (lower = better)")
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(bottom=0)
    
    # Annotate the final loss value
    axes[0].annotate(f"Final: {losses[-1]:.3f}",
                     xy=(steps[-1], losses[-1]),
                     xytext=(-60, 10), textcoords="offset points",
                     fontsize=9, color="steelblue",
                     arrowprops=dict(arrowstyle="->", color="steelblue"))
    
    # Plot 2: Validation Metrics per epoch
    epochs = list(range(1, len(eval_entries) + 1))
    colors = {"eval_f1": "green", "eval_accuracy": "orange",
              "eval_precision": "blue", "eval_recall": "red"}
    labels = {"eval_f1": "F1", "eval_accuracy": "Accuracy",
              "eval_precision": "Precision", "eval_recall": "Recall"}
    
    for key, color in colors.items():
        if key in eval_entries[0]:
            values = [e[key] for e in eval_entries]
            axes[1].plot(epochs, values, color=color, linewidth=2,
                         marker="s", markersize=6, label=labels[key])
    
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Validation Metrics per Epoch (higher = better)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xticks(epochs)
    
    plt.suptitle(f"LoRA Training (r={lora_cfg.r}, α={lora_cfg.lora_alpha}, lr={training_cfg.learning_rate})",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig("qlora_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nPlot saved as qlora_training_curves.png")
else:
    print("Not enough logged data to plot. (This is fine if training was skipped.)")

## 12. Inference

### How inference works

```
raw email string
       ↓  tokenizer()
input_ids + attention_mask  (shape: 1 × 128)
       ↓  model()
logits  (shape: 1 × 2)  →  [score_ham, score_spam]
       ↓  softmax()
probabilities  →  [prob_ham, prob_spam]
       ↓  argmax()
predicted_class  →  0 (ham) or 1 (spam)
```

Key difference from training:
- `model.eval()` — disables dropout (no random zeroing)
- `torch.no_grad()` — doesn't build the computation graph (saves memory/time)
- No `labels` in input — no loss computed, just logits

In [ ]:
def predict(text: str, model, tokenizer, device: str, max_length: int = 128) -> dict:
    """
    Runs inference on a single email string.
    
    Returns a dict with:
      prediction:   'spam' or 'ham'
      confidence:   probability of the predicted class (0–1)
      prob_ham:     probability that the email is ham
      prob_spam:    probability that the email is spam
    """
    
    # model.eval() switches to evaluation mode:
    #   - Dropout layers stop randomly zeroing values
    #   - BatchNorm layers use running statistics instead of batch statistics
    # This makes predictions deterministic (same input → same output every time).
    model.eval()
    
    # Tokenize the input text into the format the model expects.
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt"  # Return PyTorch tensors
    )
    
    # Move tensors to the same device as the model.
    # .to(device) copies the tensor to the specified device (cpu, mps, or cuda).
    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    # torch.no_grad(): tells PyTorch not to build the computation graph during this block.
    # During training, PyTorch tracks every operation to enable backpropagation.
    # At inference, we don't need gradients, so we skip this tracking.
    # Effect: ~2× faster forward pass, significantly less memory usage.
    with torch.no_grad():
        # model() runs the forward pass without computing a loss.
        # Note: we don't pass 'labels' here — without labels, Trainer doesn't
        # compute loss, just the raw logits.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    # outputs.logits shape: (1, 2) — one example, two class scores
    # .squeeze(0) removes the batch dimension: (1, 2) → (2,)
    logits = outputs.logits.squeeze(0)
    
    # torch.softmax() converts raw scores to probabilities that sum to 1.
    # Formula: softmax(z)_i = e^z_i / Σ e^z_j
    # e.g., logits=[2.1, -0.3] → softmax → [0.92, 0.08]  (92% ham, 8% spam)
    probs = torch.softmax(logits, dim=0)
    
    # .item() extracts a Python float from a scalar tensor.
    prob_ham  = probs[0].item()
    prob_spam = probs[1].item()
    
    # argmax picks the index of the highest probability.
    predicted_class = probs.argmax().item()  # 0 = ham, 1 = spam
    label = "spam" if predicted_class == 1 else "ham"
    confidence = max(prob_ham, prob_spam)
    
    return {
        "prediction": label,
        "confidence": confidence,
        "prob_ham":   prob_ham,
        "prob_spam":  prob_spam,
    }


# ── Test on some examples ──
test_emails = [
    ("SPAM", "CONGRATULATIONS!!! You have WON a $5000 prize! CLAIM NOW! Click here immediately!"),
    ("HAM",  "Hi John, just confirming our meeting at 3pm tomorrow. Please bring the Q3 report."),
    ("SPAM", "FREE iPhone 15! You are our lucky winner! Just provide your credit card to claim."),
    ("HAM",  "The pull request looks good to me. Left a few minor comments but approved overall."),
    ("SPAM", "Act now! Limited time offer! 90% discount on all medications. No prescription needed."),
    ("HAM",  "Can you review the attached proposal before Friday's client call? Thanks."),
]

print(f"{'ACTUAL':<8} {'PREDICTED':<10} {'CONF':>6}  {'P(ham)':>7}  {'P(spam)':>7}  EMAIL")
print("-" * 90)

correct = 0
for actual, email in test_emails:
    result = predict(email, model, tokenizer, device, max_length=data_cfg.max_length)
    
    # Check if prediction matches actual label
    is_correct = result["prediction"].upper() == actual
    correct += is_correct
    checkmark = "✓" if is_correct else "✗"
    
    print(f"{actual:<8} {result['prediction'].upper():<9} {checkmark} {result['confidence']:>6.1%}  "
          f"{result['prob_ham']:>7.1%}  {result['prob_spam']:>7.1%}  "
          f"{email[:55]}...")

print(f"\nAccuracy on test examples: {correct}/{len(test_emails)} = {correct/len(test_emails):.0%}")

## 13. Comparison: LoRA vs Baseline Approach

Our baseline notebook (`fine-tuning-gpt2.ipynb`) used a different strategy:
**selective layer freezing** — freeze most of GPT-2, only train the last transformer block + classification head.

| Aspect | Baseline (selective freezing) | LoRA (this notebook) |
|--------|------------------------------|---------------------|
| **Approach** | Freeze all layers, unfreeze last block | Freeze ALL weights, add adapter matrices |
| **Trainable params** | ~7M (last block + head) | ~593K (adapters + head) |
| **% of total params** | ~5.6% | ~0.5% |
| **Adapter savings** | Not saved separately | Adapter only (~5 MB vs ~500 MB) |
| **Merge-back** | N/A (just weights) | Yes: `model.merge_and_unload()` |
| **Learning rate** | 2e-5 (conservative) | 2e-4 (aggressive, safe because backbone frozen) |
| **Theory behind it** | Intuition: last layers are task-specific | Intrinsic dimensionality of fine-tuning |

LoRA has 11× fewer trainable parameters than the baseline's selective freezing approach — while adapting EVERY layer (not just the last one) through its low-rank projections.

In [ ]:
# ── Compute and visualise the parameter comparison ──

# Baseline stats (from the fine-tuning-gpt2.ipynb notebook)
# The baseline unfreezes: transformer.ln_f + transformer.h[-1] (last block) + score
total_params    = 124_443_648   # GPT-2 base model total parameters
baseline_trainable = 7_113_730  # approximate: last transformer block + head
lora_trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Parameter Comparison: Baseline vs LoRA")
print("=" * 55)
print(f"  Total params (GPT-2 base):  {total_params:>12,}")
print(f"  Baseline trainable:         {baseline_trainable:>12,}  ({100*baseline_trainable/total_params:.2f}%)")
print(f"  LoRA trainable:             {lora_trainable:>12,}  ({100*lora_trainable/total_params:.2f}%)")
print(f"  LoRA saves vs baseline:     {baseline_trainable - lora_trainable:>12,} params")
print(f"  LoRA is {baseline_trainable/lora_trainable:.1f}x more parameter-efficient than baseline")

# Visualise with a bar chart
fig, ax = plt.subplots(figsize=(10, 4))

categories = ["Full Fine-Tuning\n(hypothetical)", "Baseline\n(selective freezing)", "LoRA\n(this notebook)"]
values     = [total_params, baseline_trainable, lora_trainable]
colors     = ["#e74c3c", "#f39c12", "#27ae60"]

bars = ax.bar(categories, values, color=colors, edgecolor="white", linewidth=1.5)

# Add value labels on top of each bar
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.02,
            f"{val:,}\n({100*val/total_params:.2f}%)",
            ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Trainable Parameters")
ax.set_title("Trainable Parameters Comparison — Lower is More Efficient")
ax.set_ylim(0, total_params * 1.25)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x/1e6:.1f}M"))
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("qlora_parameter_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved as qlora_parameter_comparison.png")

## 14. Saving and Loading the Model

### What gets saved with LoRA?

With `model.save_pretrained()`, peft saves **only the adapter weights** — not the full GPT-2 model.

- `adapter_config.json` — records r, alpha, target_modules, etc.
- `adapter_model.safetensors` — the actual A and B weight tensors

These files are tiny (~5–10 MB) compared to the full model (~500 MB).

To deploy, you load the base GPT-2 from HuggingFace's hub and merge your adapter on top. You can share just the adapter file — not the full model.

In [ ]:
# Save the adapter weights and tokenizer.
save_dir = os.path.join(training_cfg.output_dir, "final_model")
os.makedirs(save_dir, exist_ok=True)

# save_pretrained() from peft: saves ONLY the LoRA adapter weights and config.
# This is fundamentally different from a standard HuggingFace save_pretrained()
# which would save all model weights.
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Let's check what was saved and how large it is.
print(f"Files saved to: {save_dir}/")
for filename in sorted(os.listdir(save_dir)):
    filepath = os.path.join(save_dir, filename)
    size_kb = os.path.getsize(filepath) / 1024
    if size_kb > 1024:
        print(f"  {filename:<40} {size_kb/1024:.1f} MB")
    else:
        print(f"  {filename:<40} {size_kb:.1f} KB")

print(f"\nFor comparison: full GPT-2 model = ~500 MB")
total_size = sum(os.path.getsize(os.path.join(save_dir, f)) for f in os.listdir(save_dir))
print(f"LoRA adapter total size: {total_size / 1e6:.1f} MB")

In [ ]:
# ── Reload the saved adapter and run inference ──
# This demonstrates the loading pattern for deployment / sharing.

from peft import PeftModel
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer

print("Loading model from saved adapter...")

# Step 1: Load the frozen base model from HuggingFace.
# This is the unmodified pre-trained GPT-2 — 124M parameters, no fine-tuning.
base_model_fresh = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
base_model_fresh.config.pad_token_id = base_model_fresh.config.eos_token_id
print("  Base GPT-2 loaded (frozen, unmodified)")

# Step 2: Overlay the LoRA adapter on top of the base model.
# PeftModel.from_pretrained() reads adapter_config.json to know where
# to attach the adapters, then loads the weight values from adapter_model.safetensors.
loaded_model = PeftModel.from_pretrained(base_model_fresh, save_dir)
print(f"  LoRA adapter loaded from: {save_dir}")

# Step 3: Load the tokenizer.
loaded_tokenizer = GPT2Tokenizer.from_pretrained(save_dir)
loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

# Step 4: Move to device (the loaded model starts on CPU).
loaded_model = loaded_model.to(device)

print("\nVerification — running inference with reloaded model:")
test_text = "URGENT: Your account has been compromised! Click here to verify NOW!"
result = predict(test_text, loaded_model, loaded_tokenizer, device)
print(f"  Input: '{test_text[:60]}...'")
print(f"  Prediction: {result['prediction'].upper()}  (confidence: {result['confidence']:.1%})")
print(f"  P(ham)={result['prob_ham']:.1%}  P(spam)={result['prob_spam']:.1%}")

## 15. Summary and Next Steps

### What you built

A spam classifier that:
- Uses GPT-2 (124M params) as the backbone
- Fine-tunes **only ~0.5%** of those parameters using LoRA
- Saves a ~5 MB adapter (vs ~500 MB for full fine-tuning)
- Achieves comparable accuracy to full fine-tuning

### What you learned

| Concept | Where to find it |
|---------|------------------|
| LoRA math (rank, alpha, A/B matrices) | Section 2 + `src/qlora/model.py` |
| QLoRA quantization | Section 3 |
| Tokenization (BPE, padding, attention mask) | Section 8 + `src/qlora/dataset.py` |
| Classification loss (CrossEntropy) | Section 10 + `src/qlora/trainer.py` |
| F1 vs accuracy | Section 11 + `src/qlora/trainer.py` |
| Lazy dataset loading pattern | `src/qlora/dataset.py` |
| HuggingFace Trainer internals | `src/qlora/trainer.py` |

### Experiments to try next

**1. Rank ablation** — how does performance change with different ranks?
```python
for r in [4, 8, 16, 32]:
    results = train(lora_config=LoRAConfig(r=r, lora_alpha=r*2))
    print(f"r={r}: F1={results['val_f1']:.4f}")
```

**2. Target module ablation** — does adapting FFN layers help?
```python
# Attention only (current)
train(lora_config=LoRAConfig(target_modules=["c_attn", "c_proj"]))
# Attention + FFN
train(lora_config=LoRAConfig(target_modules=["c_attn", "c_proj", "c_fc"]))
```

**3. Scale to a larger model** — try GPT-2 medium or large:
```python
tokenizer = create_tokenizer("gpt2-medium")  # 345M params
model = create_model(lora_cfg, model_name="gpt2-medium")
```

**4. True QLoRA on Colab** — add 4-bit quantization (requires CUDA):
```python
from transformers import BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16)
base = GPT2ForSequenceClassification.from_pretrained("gpt2", quantization_config=bnb)
```

**5. Different dataset** — try sentiment analysis or topic classification:
```python
from datasets import load_dataset
dataset = load_dataset("imdb")  # sentiment: positive/negative movie reviews
```

---
*Built as a portfolio learning project. Each source file in `src/qlora/` contains detailed explanations of every line.*